# Workflow patterns

Assistant version 9: Map-reduce product reviews by category and improve the report with a bounded evaluator loop.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra deep --group authoring`, then `uv run --extra deep --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Prompt chaining; validation gates

Predict the output before running. The function definitions below match `examples/01_chain_gate.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Chain a draft and revision behind a code validation gate.

Run: uv run --extra agents python modules/15_workflows/examples/01_chain_gate.py --offline
Expected: A missing citation blocks publication; a revision supplies it.
Cost: At most two short model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


class State(TypedDict):
    draft: str
    published: bool


def main() -> None:
    settings = lesson_settings()
    model = get_chat_model(
        settings=settings,
        responses=[
            "Synthetic review: setup matters.",
            "Synthetic review: setup matters [data/reviews.csv].",
        ],
    )

    def draft(state: State) -> dict:
        return {
            "draft": str(model.invoke("Draft one cautious insight from synthetic reviews.").content)
        }

    def gate(state: State) -> Literal["publish", "revise"]:
        return "publish" if "data/reviews.csv" in state["draft"] else "revise"

    def revise(state: State) -> dict:
        return {
            "draft": str(
                model.invoke(
                    "Add source data/reviews.csv to this synthetic draft: " + state["draft"]
                ).content
            )
        }

    def publish(state: State) -> dict:
        # KEY IDEA: prompts request a citation; code refuses an uncited final draft.
        if "data/reviews.csv" not in state["draft"]:
            raise ValueError("Citation still missing after the single revision.")
        return {"published": True}

    graph = StateGraph(State)
    for name, node in [("draft", draft), ("revise", revise), ("publish", publish)]:
        graph.add_node(name, node)
    graph.add_edge(START, "draft")
    graph.add_conditional_edges("draft", gate)
    graph.add_edge("revise", "publish")
    graph.add_edge("publish", END)
    print_text(str(graph.compile().invoke({"draft": "", "published": False})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Routing; explicit fallback branch

Predict the output before running. The function definitions below match `examples/02_route.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Route validated intent with a visible fallback.

Run: uv run --extra agents python modules/15_workflows/examples/02_route.py --offline
Expected: Shipping selects logistics; an unknown request selects manual review.
Cost: No model calls; intent labels are explicit inputs. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


class State(TypedDict):
    intent: str
    queue: str


def route(state: State) -> Literal["logistics", "billing", "fallback"]:
    return {"shipping": "logistics", "refund": "billing"}.get(state["intent"], "fallback")


def main() -> None:
    lesson_settings()
    graph = StateGraph(State)
    graph.add_node("logistics", lambda state: {"queue": "delivery desk"})
    graph.add_node("billing", lambda state: {"queue": "refund review"})
    graph.add_node("fallback", lambda state: {"queue": "manual clarification"})
    # KEY IDEA: an unknown label is not permission to pick a privileged branch.
    graph.add_conditional_edges(START, route)
    for node in ("logistics", "billing", "fallback"):
        graph.add_edge(node, END)
    workflow = graph.compile()
    for intent in ("shipping", "unknown"):
        print_text(str(workflow.invoke({"intent": intent, "queue": ""})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Fan-out/fan-in; reducers; deterministic merge

Predict the output before running. The function definitions below match `examples/03_parallel.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Join two independent review analyses before producing a summary.

Run: uv run --extra agents python modules/15_workflows/examples/03_parallel.py --offline
Expected: One deterministic report contains both worker outputs.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.workflows import category_facts, review_groups


class State(TypedDict):
    facts: Annotated[list[dict], operator.add]
    report: str


def ratings(state: State) -> dict:
    groups = review_groups()
    category = next(iter(groups))
    return {"facts": [{"kind": "ratings", "value": category_facts(category, groups[category])}]}


def sample(state: State) -> dict:
    return {"facts": [{"kind": "scope", "value": "Synthetic reviews, not market research."}]}


def merge(state: State) -> dict:
    return {"report": str(sorted(state["facts"], key=lambda row: row["kind"]))}


def main() -> None:
    lesson_settings()
    graph = StateGraph(State)
    for name, node in [("ratings", ratings), ("sample", sample), ("merge", merge)]:
        graph.add_node(name, node)
    graph.add_edge(START, "ratings")
    graph.add_edge(START, "sample")
    # KEY IDEA: a list edge waits for both producers; the reducer accepts their writes.
    graph.add_edge(["ratings", "sample"], "merge")
    graph.add_edge("merge", END)
    print_text(str(graph.compile().invoke({"facts": [], "report": ""})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Send API; dynamic map-reduce

Predict the output before running. The function definitions below match `examples/04_send_map_reduce.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Create a dynamic worker per review category with Send.

Run: uv run --extra agents python modules/15_workflows/examples/04_send_map_reduce.py --offline
Expected: Six category groups are reduced in sorted order.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.workflows import category_facts, review_groups


class State(TypedDict):
    categories: list[str]
    facts: Annotated[list[dict], operator.add]
    report: str


class Work(TypedDict):
    category: str


def dispatch(state: State) -> list[Send]:
    if not state["categories"] or len(set(state["categories"])) != len(state["categories"]):
        raise ValueError("Use nonempty unique categories.")
    return [Send("analyze", {"category": category}) for category in state["categories"]]


def analyze(state: Work) -> dict:
    return {"facts": [category_facts(state["category"], review_groups()[state["category"]])]}


def main() -> None:
    lesson_settings()
    graph = StateGraph(State)
    graph.add_node("analyze", analyze)
    graph.add_node(
        "merge",
        lambda state: {"report": str(sorted(state["facts"], key=lambda row: row["category"]))},
    )
    # KEY IDEA: each Send contains only the worker's input; results enter a shared reducer.
    graph.add_conditional_edges(START, dispatch, ["analyze"])
    graph.add_edge("analyze", "merge")
    graph.add_edge("merge", END)
    print_text(
        str(
            graph.compile().invoke({"categories": list(review_groups()), "facts": [], "report": ""})
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Orchestrator planning; workers; result aggregation

Predict the output before running. The function definitions below match `examples/05_orchestrator_workers.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Validate a model plan before dispatching bounded workers.

Run: uv run --extra agents python modules/15_workflows/examples/05_orchestrator_workers.py --offline
Expected: A real structured parser produces two allowed category assignments.
Cost: One short planning model call. Offline fixtures incur no API fee.
Requires: agents
"""

import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field

from common.errors import run_cli
from common.fixtures import structured_fixture
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.workflows import category_facts, review_groups


class Plan(BaseModel):
    categories: list[str] = Field(min_length=1, max_length=3)


class State(TypedDict):
    categories: list[str]
    facts: Annotated[list[dict], operator.add]


def main() -> None:
    settings = lesson_settings()
    groups = review_groups()
    model = get_chat_model(
        settings=settings, responses=[structured_fixture(Plan, {"categories": list(groups)[:2]})]
    ).with_structured_output(Plan)

    def plan(state: State) -> dict:
        result = model.invoke(
            "Select up to three categories for a review audit: " + str(list(groups))
        )
        if (
            len(set(result.categories)) != len(result.categories)
            or set(result.categories) - groups.keys()
        ):
            raise ValueError("Planner returned duplicate or unknown work.")
        return {"categories": result.categories}

    def dispatch(state: State) -> list[Send]:
        return [Send("worker", {"category": category}) for category in state["categories"]]

    def worker(state: dict) -> dict:
        return {"facts": [category_facts(state["category"], groups[state["category"]])]}

    # KEY IDEA: the planner proposes a finite plan; validated code controls dispatch.
    graph = StateGraph(State)
    graph.add_node("plan", plan)
    graph.add_node("worker", worker)
    graph.add_edge(START, "plan")
    graph.add_conditional_edges("plan", dispatch, ["worker"])
    graph.add_edge("worker", END)
    result = graph.compile().invoke({"categories": [], "facts": []})
    print_text(str(sorted(result["facts"], key=lambda row: row["category"])))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Evaluator-optimizer; quality rubric; iteration/cost ceiling

Predict the output before running. The function definitions below match `examples/06_evaluator_optimizer.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Improve a draft until the gate passes or the iteration budget ends.

Run: uv run --extra agents python modules/15_workflows/examples/06_evaluator_optimizer.py --offline
Expected: Two attempts, six logical model calls, and an accepted synthetic report.
Cost: At most six short calls: two workers plus two draft/evaluate pairs. Offline fixtures incur
no API fee.
Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.workflows import build_review_graph, review_groups


def main() -> None:
    settings = lesson_settings()
    categories = list(review_groups())[:2]
    # KEY IDEA: maximum attempts is enforced even if an evaluator never approves.
    result = build_review_graph(settings, max_attempts=2).invoke(
        {"categories": categories, "sections": [], "attempt": 0, "model_calls": 0},
        {"recursion_limit": 12, "max_concurrency": 2},
    )
    print_text(
        str({key: result[key] for key in ("attempt", "model_calls", "accepted", "feedback")})
    )
    print_text(result["report"])


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Fixed workflow versus autonomous agent; compare measurable outcomes

Predict the output before running. The function definitions below match `examples/07_choose_workflow.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare a fixed read with agent-driven tool selection.

Run: uv run --extra agents python modules/15_workflows/examples/07_choose_workflow.py --offline
Expected: Both get the same stock; the fixed read uses zero model calls.
Cost: At most three short agent model calls; fixed path has no model fee. Offline fixtures incur
no API fee.
Requires: agents
"""

from time import perf_counter

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    stock = make_order_tools("C001")[2]
    start = perf_counter()
    fixed = stock.invoke({"sku": "BW-001"})
    fixed_ms = (perf_counter() - start) * 1000
    model = get_chat_model(
        settings=settings,
        responses=[tool_request("check_stock", {"sku": "BW-001"}), "Read complete."],
    )
    agent = create_agent(
        model, [stock], middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")]
    )
    start = perf_counter()
    state = agent.invoke({"messages": [{"role": "user", "content": "Check BW-001 stock."}]})
    # KEY IDEA: compare observed work, not an invented claim that agents are more accurate.
    print_text(
        str(
            {
                "fixed_result": fixed,
                "fixed_ms": round(fixed_ms, 2),
                "agent_ms": round((perf_counter() - start) * 1000, 2),
                "agent_model_turns": sum(m.type == "ai" for m in state["messages"]),
                "agent_tool_results": [m.content for m in state["messages"] if m.type == "tool"],
            }
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Map-reduce product reviews by category and improve the report with a bounded evaluator loop

Predict the output before running. The function definitions below match `use_case/weekly_report.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Map all review categories, evaluate a draft, and save only an accepted report.

Run: uv run --extra agents python modules/15_workflows/use_case/weekly_report.py --offline
Expected: All categories covered; two attempts; report saved under the runtime directory.
Cost: At most ten short calls with six categories. Requires: agents
"""

import json

from common.config import ROOT
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.workflows import build_review_graph, review_groups


def main() -> None:
    settings = lesson_settings()
    categories = list(review_groups())
    result = build_review_graph(settings).invoke(
        {"categories": categories, "sections": [], "attempt": 0, "model_calls": 0},
        {"max_concurrency": 2, "recursion_limit": 12},
    )
    output = ROOT / settings.runtime_dir / "weekly-report"
    output.mkdir(parents=True, exist_ok=True)
    # KEY IDEA: a rejected draft remains diagnostic evidence; it is not a published report.
    (output / "evaluation.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    if result["accepted"]:
        (output / "report.md").write_text(result["report"], encoding="utf-8")
    print_text(
        str(
            {
                "accepted": result["accepted"],
                "attempts": result["attempt"],
                "model_calls": result["model_calls"],
                "output": str(output),
            }
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra agents pytest modules/15_workflows/tests` for the supplied solution checks.